Exercice 1

In [2]:
import sqlalchemy
from sqlalchemy.orm import sessionmaker
from sqlalchemy.ext.automap import automap_base

# Create engine and connection
engine = sqlalchemy.create_engine('sqlite:///chinook.db')
cur = engine.connect()

# Reflect the database
metadata = sqlalchemy.MetaData()
metadata.reflect(engine)

# Automap base
Base = automap_base(metadata=metadata)
Base.prepare()

# Create a session
Session = sessionmaker(bind=engine)
session = Session()

Exercice 2

In [4]:
from sqlalchemy import inspect

# Use inspector to get table names
inspector = inspect(engine)
table_names = inspector.get_table_names()
print("Tables in the database:", table_names)


Tables in the database: ['albums', 'artists', 'customers', 'employees', 'genres', 'invoice_items', 'invoices', 'media_types', 'playlist_track', 'playlists', 'tracks']


Exercice 3

In [6]:
Track = Base.classes.tracks

# Query the first 3 tracks
tracks = session.query(Track).limit(3).all()

# Display the names
for t in tracks:
    print(t.Name)

For Those About To Rock (We Salute You)
Balls to the Wall
Fast As a Shark


Exercice 4

In [8]:
Track = Base.classes.tracks
Album = Base.classes.albums

# Query the first 20 tracks with their albums using a join
results = session.query(Track.Name, Album.Title).join(Album, Track.AlbumId == Album.AlbumId).limit(20).all()

# Print results
for track_name, album_title in results:
    print(f"Track: {track_name} - Album: {album_title}")

Track: For Those About To Rock (We Salute You) - Album: For Those About To Rock We Salute You
Track: Put The Finger On You - Album: For Those About To Rock We Salute You
Track: Let's Get It Up - Album: For Those About To Rock We Salute You
Track: Inject The Venom - Album: For Those About To Rock We Salute You
Track: Snowballed - Album: For Those About To Rock We Salute You
Track: Evil Walks - Album: For Those About To Rock We Salute You
Track: C.O.D. - Album: For Those About To Rock We Salute You
Track: Breaking The Rules - Album: For Those About To Rock We Salute You
Track: Night Of The Long Knives - Album: For Those About To Rock We Salute You
Track: Spellbound - Album: For Those About To Rock We Salute You
Track: Balls to the Wall - Album: Balls to the Wall
Track: Fast As a Shark - Album: Restless and Wild
Track: Restless and Wild - Album: Restless and Wild
Track: Princess of the Dawn - Album: Restless and Wild
Track: Go Down - Album: Let There Be Rock
Track: Dog Eat Dog - Album: Le

Exercice 5

In [12]:
InvoiceItem = Base.classes.invoice_items
Track = Base.classes.tracks

results = session.query(Track.Name, InvoiceItem.Quantity)\
    .join(Track, InvoiceItem.TrackId == Track.TrackId)\
    .limit(10).all()

for track_name, quantity in results:
    print(f"Track: {track_name} - Quantity Sold: {quantity}")

Track: Balls to the Wall - Quantity Sold: 1
Track: Restless and Wild - Quantity Sold: 1
Track: Put The Finger On You - Quantity Sold: 1
Track: Inject The Venom - Quantity Sold: 1
Track: Evil Walks - Quantity Sold: 1
Track: Breaking The Rules - Quantity Sold: 1
Track: Dog Eat Dog - Quantity Sold: 1
Track: Overdose - Quantity Sold: 1
Track: Love In An Elevator - Quantity Sold: 1
Track: Janie's Got A Gun - Quantity Sold: 1


In [13]:
print(Base.classes.keys())

['customers', 'artists', 'genres', 'tracks', 'invoice_items', 'invoices', 'employees', 'albums', 'playlists', 'media_types']


Exercice 6

In [14]:
InvoiceItem = Base.classes.invoice_items
Track = Base.classes.tracks

from sqlalchemy import func

# Aggregate sales quantity per track and order descending
top_tracks = session.query(
    Track.Name,
    func.sum(InvoiceItem.Quantity).label('total_sold')
).join(Track, InvoiceItem.TrackId == Track.TrackId)\
.group_by(Track.Name)\
.order_by(func.sum(InvoiceItem.Quantity).desc())\
.limit(10).all()

# Print results
for track_name, total_sold in top_tracks:
    print(f"Track: {track_name} - Times Sold: {total_sold}")

Track: The Trooper - Times Sold: 5
Track: Untitled - Times Sold: 4
Track: The Number Of The Beast - Times Sold: 4
Track: Sure Know Something - Times Sold: 4
Track: Hallowed Be Thy Name - Times Sold: 4
Track: Eruption - Times Sold: 4
Track: Where Eagles Dare - Times Sold: 3
Track: Welcome Home (Sanitarium) - Times Sold: 3
Track: Sweetest Thing - Times Sold: 3
Track: Surrender - Times Sold: 3


Exercice 7

In [15]:
Artist = Base.classes.artists
Album = Base.classes.albums
Track = Base.classes.tracks
InvoiceItem = Base.classes.invoice_items

from sqlalchemy import func

# Join tables: InvoiceItem → Track → Album → Artist
top_artists = session.query(
    Artist.Name,
    func.sum(InvoiceItem.Quantity).label('total_sold')
).join(Track, InvoiceItem.TrackId == Track.TrackId)\
 .join(Album, Track.AlbumId == Album.AlbumId)\
 .join(Artist, Album.ArtistId == Artist.ArtistId)\
 .group_by(Artist.Name)\
 .order_by(func.sum(InvoiceItem.Quantity).desc())\
 .limit(10).all()

# Print results
for artist_name, total_sold in top_artists:
    print(f"Artist: {artist_name} - Total Tracks Sold: {total_sold}")

Artist: Iron Maiden - Total Tracks Sold: 140
Artist: U2 - Total Tracks Sold: 107
Artist: Metallica - Total Tracks Sold: 91
Artist: Led Zeppelin - Total Tracks Sold: 87
Artist: Os Paralamas Do Sucesso - Total Tracks Sold: 45
Artist: Deep Purple - Total Tracks Sold: 44
Artist: Faith No More - Total Tracks Sold: 42
Artist: Lost - Total Tracks Sold: 41
Artist: Eric Clapton - Total Tracks Sold: 40
Artist: R.E.M. - Total Tracks Sold: 39
